# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset is structured according to the Croissant schema and is available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

This dataset provides case-level tabulations on clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. The Croissant schema URL is specified below.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display main metadata elements
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id` identifiers. This helps us understand the dataset contents and structure.

Here, we will list record sets found in the metadata, then for each record set, display sample records and field (column) IDs.

In [ ]:
# List available record sets and preview their content
record_sets = []

# The metadata.recordSet attribute may be empty if not parsed, so we use dataset.record_sets() for discovery.
for record_set in dataset.record_sets():  # Each record set is a mlcroissant RecordSet object
    print(f"Record Set Name: {record_set.name} (@id: {record_set.id})")
    record_sets.append(record_set.id)
    print("Field @ids and Names:")
    for field in record_set.fields:
        print(f"  - {field.id}: {field.name}")
    # Show a preview of records
    preview = list(dataset.records(record_set=record_set.id))
    print(f"--- Sample Record (first row):")
    if preview:
        print({k: v for k, v in preview[0].items()})
    print()
print(f"Discovered {len(record_sets)} record sets:\n{record_sets}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis. Use the record set and field `@id`s from the overview above.

All references to record sets and fields use their `@id`s.

In [ ]:
# Extract all records from each record set into a DataFrame
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set '@id': {record_set_id}")
    print(f"Columns (@ids): {df.columns.tolist()}")
    print(df.head(), "\n")

# For demonstration, pick the first record set for analysis
selected_record_set_id = record_sets[0] if record_sets else None
if selected_record_set_id:
    print(f"Columns in '{selected_record_set_id}': {dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering records by field values, normalizing numeric fields, and grouping/categorizing data.

All fields are referenced by their `@id`. Adjust the field @id below to match one suitable from your overview above.

In [ ]:
# Example: Select a numeric column for analysis
numeric_field_id = None
group_field_id = None

# Try to find the first numeric field in the dataframe (float/int)
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    for col in df.columns:
        # Attempt to detect numeric fields
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Example: Group by a categorical field (pick second field that's not numeric)
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break

    if numeric_field_id:
        # Filter records: e.g., numeric_field > 10
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())
        
        # Normalize numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by another field and compute mean
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets found to perform EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Below is an example using matplotlib for numeric field visualization. Adjust the field @ids as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution if available
if selected_record_set_id and numeric_field_id:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouping field exists
    if group_field_id:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped: No numeric or grouping fields found.")

## 6. Conclusion
This notebook demonstrated loading and exploring the Genotype–Phenotype Heterogeneity dataset using `mlcroissant`, referencing all entities by their `@id` for transparency and reproducibility.

**Key points:**
- Croissant metadata reveals structured record sets and fields, each with semantic `@id` identifiers.
- Data from all record sets is loadable and analyzable via `mlcroissant`.
- Exploratory and visualization steps help uncover clinical, hormonal, or genetic patterns.
- For further research or clinical replication, reference field and record set `@id`s to precisely extract relevant variables.

For advanced tasks, see `mlcroissant` documentation and FAIR^2 platform for dataset citation, provenance, and sharing.